In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path.home() / "Downloads" / "bluestock capstone dataset"
NAV_RAW_PATH = DATA_DIR / "02_nav_history.csv"

Nav_history cleaning


In [4]:
WORKSPACE_DIR = Path(r"C:\Users\P LIKITH VARMA\OneDrive\Desktop\mutual_fund_analtics\mutual_fund_analytics_platform")
PROCESSED_DIR = WORKSPACE_DIR / "data" / "processed"
OUTPUT_CLEAN_PATH = PROCESSED_DIR / "clean_nav.csv"

In [5]:
OUTPUT_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

print("⏳ Loading raw NAV history data...")
df_nav = pd.read_csv(NAV_RAW_PATH)
print(f"✔ Initial raw row count: {len(df_nav)}")

⏳ Loading raw NAV history data...
✔ Initial raw row count: 46000


In [6]:
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

In [7]:
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])
df_nav = df_nav[df_nav['nav'] > 0]

In [8]:
def fill_missing_dates(group):
    group = group.set_index('date')
    # Create complete calendar daily range from the fund's start to end date
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group = group.reindex(full_range)
    
    # Forward fill weekend/holiday gaps with the previous business day's price
    group['nav'] = group['nav'].ffill()
    group['amfi_code'] = group['amfi_code'].ffill().astype(int)
    return group.reset_index().rename(columns={'index': 'date'})

print("⏳ Forward-filling missing timeline gaps (weekends & market holidays)...")
df_clean_nav = df_nav.groupby('amfi_code', group_keys=False).apply(fill_missing_dates)
df_clean_nav = df_clean_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

print(f"✔ Final cleaned row count (expanded with calendar days): {len(df_clean_nav)}")

⏳ Forward-filling missing timeline gaps (weekends & market holidays)...
✔ Final cleaned row count (expanded with calendar days): 64320


C:\Users\P LIKITH VARMA\AppData\Local\Temp\ipykernel_4992\2930567328.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean_nav = df_nav.groupby('amfi_code', group_keys=False).apply(fill_missing_dates)


In [9]:
df_clean_nav.to_csv(OUTPUT_CLEAN_PATH, index=False)
print(f"💾 Successfully saved output to: {OUTPUT_CLEAN_PATH}")

💾 Successfully saved output to: C:\Users\P LIKITH VARMA\OneDrive\Desktop\mutual_fund_analtics\mutual_fund_analytics_platform\data\processed\clean_nav.csv


Investors Transactions Cleaning


In [22]:
import re
TRANS_RAW_PATH = DATA_DIR / "08_investor_transactions.csv"

In [11]:
OUTPUT_CLEAN_PATH = PROCESSED_DIR / "clean_investor_transactions.csv"

In [24]:
OUTPUT_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

print("⏳ Loading raw investor transaction data...")
df_tx = pd.read_csv(TRANS_RAW_PATH)
print(f"✔ Initial raw transaction count: {len(df_tx)}")

⏳ Loading raw investor transaction data...
✔ Initial raw transaction count: 32778


In [17]:
# --- Step 1: Standardize transaction_type using Regular Expressions (re) ---
def clean_tx_type(text):
    if not isinstance(text, str):
        return "Unknown"
    
    # Strip whitespace and make lowercase for robust matching
    text_clean = text.strip().lower()
    
    # Use regex patterns to group messy variations
    if re.search(r'sip|systematic', text_clean):
        return 'SIP'
    elif re.search(r'lump|one-time|single', text_clean):
        return 'Lumpsum'
    elif re.search(r'redemp|sell|withdraw', text_clean):
        return 'Redemption'
    else:
        return 'Other'

print("⏳ Standardizing transaction types using regex...")
df_tx['transaction_type'] = df_tx['transaction_type'].apply(clean_tx_type)

⏳ Standardizing transaction types using regex...


In [18]:
print("⏳ Validating transaction amounts...")
df_tx = df_tx[df_tx['amount_inr'] > 0]

⏳ Validating transaction amounts...


In [19]:
print("⏳ Standardizing KYC status labels...")
df_tx['kyc_status'] = df_tx['kyc_status'].astype(str).str.strip().str.capitalize()

⏳ Standardizing KYC status labels...


In [20]:
print("⏳ Converting transaction dates to standard datetime format...")
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

⏳ Converting transaction dates to standard datetime format...


In [21]:
df_tx = df_tx.sort_values(by=['investor_id', 'transaction_date']).reset_index(drop=True)
print(f"✔ Final cleaned transaction count: {len(df_tx)}")

# 2. Save the clean file directly into your target processed folder
df_tx.to_csv(OUTPUT_CLEAN_PATH, index=False)
print(f"💾 Successfully saved transaction output to: {OUTPUT_CLEAN_PATH}")

✔ Final cleaned transaction count: 32778
💾 Successfully saved transaction output to: C:\Users\P LIKITH VARMA\OneDrive\Desktop\mutual_fund_analtics\mutual_fund_analytics_platform\data\processed\clean_investor_transactions.csv


Scheme Performance Cleaning


In [30]:
PERF_RAW_PATH = DATA_DIR / "07_scheme_performance.csv"

In [31]:
OUTPUT_CLEAN_PATH = PROCESSED_DIR / "clean_scheme_performance.csv"

In [32]:
OUTPUT_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

print("⏳ Loading raw scheme performance data...")
df_perf = pd.read_csv(PERF_RAW_PATH)
print(f"✔ Initial raw records count: {len(df_perf)}")

⏳ Loading raw scheme performance data...
✔ Initial raw records count: 40


In [33]:
# --- Step 1: Validate return values are numeric ---
print("⏳ Verifying all financial return values are numeric...")
# Identify columns that should be strictly numeric based on your data schema
numeric_cols = [
    'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 
    'benchmark_3yr_pct', 'alpha', 'beta', 
    'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct'
]

for col in numeric_cols:
    if col in df_perf.columns:
        # errors='coerce' turns any unparseable string (like text or blanks) into NaN
        df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')
        # Fill any unexpected blank NaNs with 0.0 to avoid breaking future math
        df_perf[col] = df_perf[col].fillna(0.0)

⏳ Verifying all financial return values are numeric...


In [34]:
# --- Step 2: Flag negative Sharpe Ratios ---
print("⏳ Labeling and flagging negative Sharpe ratios...")
# We create a new boolean flag column: True if risk-adjusted return is negative, False otherwise
df_perf['is_negative_sharpe'] = df_perf['sharpe_ratio'] < 0

⏳ Labeling and flagging negative Sharpe ratios...


In [35]:
# --- Step 3: Check expense_ratio range (0.1% to 2.5%) ---
print("⏳ Auditing expense ratios against SEBI thresholds (0.1% - 2.5%)...")
# Find rows out of bounds just to log them, then clamp them to realistic boundaries
out_of_bounds = df_perf[(df_perf['expense_ratio_pct'] < 0.1) | (df_perf['expense_ratio_pct'] > 2.5)]
if not out_of_bounds.empty:
    print(f"⚠️ Found {len(out_of_bounds)} funds with unusual expense ratios. Correcting paths...")
    # Clean up by clipping values strictly to the legal 0.1% - 2.5% boundary
    df_perf['expense_ratio_pct'] = df_perf['expense_ratio_pct'].clip(lower=0.1, upper=2.5)

⏳ Auditing expense ratios against SEBI thresholds (0.1% - 2.5%)...


In [36]:
df_perf = df_perf.sort_values(by='amfi_code').reset_index(drop=True)

# 2. Save the clean file directly into your target processed folder
df_perf.to_csv(OUTPUT_CLEAN_PATH, index=False)
print(f"💾 Successfully saved performance analytics output to: {OUTPUT_CLEAN_PATH}")

💾 Successfully saved performance analytics output to: C:\Users\P LIKITH VARMA\OneDrive\Desktop\mutual_fund_analtics\mutual_fund_analytics_platform\data\processed\clean_scheme_performance.csv
